# Project 5: Customer Segmentation
**Задача:** разбить клиентов магазина на сегменты БЕЗ ответов (unsupervised, K-Means + PCA).
**Темы:** clustering, elbow, интерпретация сегментов, PCA-визуализация.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

rng = np.random.default_rng(5)
def group(center, n):
    return rng.normal(center, [8, 9, 1.5, 4], (n, 4))
# 4 скрытых типа клиентов (модель об этом НЕ знает)
data = np.vstack([
    group([25, 20, 2, 8],  120),   # студенты: мало тратят, часто заходят
    group([85, 75, 8, 25], 100),   # VIP: много тратят, крупный чек
    group([70, 25, 3, 30], 110),   # богатые экономные
    group([40, 60, 6, 12], 90),    # активные средние
]).clip(0)
df = pd.DataFrame(data, columns=["доход_тыс", "траты_тыс", "покупок_в_мес", "ср_чек_тыс"]).round(1)
print(df.shape); df.head()

In [ ]:
# EDA + scaling (для KMeans обязателен!)
print(df.describe().round(1))
X_s = StandardScaler().fit_transform(df)

# Elbow: выбираем K
inertias = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_s).inertia_ for k in range(1, 10)]
plt.plot(range(1, 10), inertias, 'o-'); plt.xlabel('K'); plt.ylabel('inertia')
plt.title('Elbow: ищем локоть'); plt.grid(True); plt.show()

In [ ]:
# Кластеризация с выбранным K и интерпретация сегментов
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(X_s)
df["сегмент"] = km.labels_

profile = df.groupby("сегмент").mean().round(1)
profile["размер"] = df["сегмент"].value_counts().sort_index()
print("Портрет сегментов (средние значения):")
print(profile)
# Дай сегментам ИМЕНА по этой таблице — это и есть главная работа аналитика!

In [ ]:
# PCA: рисуем 4-мерных клиентов на плоскости
pts = PCA(n_components=2).fit_transform(X_s)
plt.figure(figsize=(8, 6))
sc = plt.scatter(pts[:, 0], pts[:, 1], c=df["сегмент"], cmap='tab10', s=15)
plt.colorbar(sc, label='сегмент'); plt.title('Сегменты клиентов (PCA 4D -> 2D)'); plt.grid(True); plt.show()

## Выводы
- Без единого ответа y модель нашла структуру: elbow указал на K≈4 — ровно столько скрытых групп мы заложили.
- Таблица-портрет — главный результат для бизнеса: «сегмент 1 = VIP (доход 85, траты 75) → персональный менеджер»,
  «сегмент 0 = студенты → промокоды» и т.д.
- PCA показал: сегменты — реальные сгустки, а не выдумка алгоритма.

## Задания
1. Запусти с K=2 и K=8. Какие сегменты «слиплись», какие раздробились?
2. Придумай маркетинговое действие под каждый из 4 сегментов и впиши в notebook.
3. Прогони IsolationForest — есть ли клиенты-аномалии вне всех сегментов?